# Cosmological Exterior (Slice 5 of Phase 2C)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bshepp/alcubierre/blob/main/cosmological_exterior.ipynb)

**Runtime:** local (all cells; symbolic + light numerics).

Slice 5 of Phase 2C. Tests whether replacing the asymptotically flat vacuum exterior with an FRW + $\Lambda$ background opens any loophole in the Package 3 acceleration result.

**Goal.** Package 3 used asymptotic flatness (ADM 4-momentum is conserved at spatial infinity). The real universe is FRW with dark-energy density $\rho_\Lambda \sim 10^{-29}\,\mathrm{g/cm^3}$. **Question**: does the cosmological background provide enough "exterior matter" for the Mechanism A loophole (push-from-a-wall), and if so, what $\Delta v$ ceiling does this set?

**Method.** Use the McVittie metric for a static central mass embedded in an FRW background. Compute the quasi-local Brown-York momentum at finite radius $R_{\rm BY} \gg R_{\rm shell}$ and ask how much momentum the dark-energy background can transfer to the shell over its lifetime.

**Conventions.** Geometrised units except where noted; SI conversions for dark-energy magnitudes.

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules or os.environ.get("HF_JOB"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import math, time
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, Symbol, Function, Matrix, simplify, lambdify, sqrt, exp, Rational
%matplotlib inline
print("sympy", sp.__version__)

sympy 1.14.0


## Cell 1 — McVittie metric

The McVittie metric (1933) describes a static central mass $m$ in an FRW background with scale factor $a(t)$. In isotropic coordinates:

$$ds^2 = -\frac{(1 - \tilde m/2\bar r)^2}{(1 + \tilde m/2\bar r)^2}\,dt^2 + (1 + \tilde m/2\bar r)^4\,a(t)^2\bigl(d\bar r^2 + \bar r^2\,d\Omega^2\bigr),$$

with $\tilde m = m/a(t)$ chosen so that the *physical* mass $m$ is constant.

For pure $\Lambda$ background: $a(t) = e^{Ht}$ with Hubble rate $H = \sqrt{\Lambda/3}$. We compute the Einstein tensor symbolically and check that at large $\bar r$ it reduces to the FRW stress-energy ($\rho = \Lambda/8\pi$, $p = -\Lambda/8\pi$).

In [2]:
t, r, theta, phi = symbols('t r theta phi', real=True, positive=True)
m, H = symbols('m H', positive=True)
a_t = sp.exp(H * t)
m_tilde = m / a_t

g_tt = -((1 - m_tilde/(2*r))**2) / ((1 + m_tilde/(2*r))**2)
g_rr = (1 + m_tilde/(2*r))**4 * a_t**2
g_thth = g_rr * r**2
g_phph = g_rr * r**2 * sp.sin(theta)**2
g = Matrix([
    [g_tt,    0, 0, 0],
    [0, g_rr, 0, 0],
    [0, 0, g_thth, 0],
    [0, 0, 0, g_phph],
])
g_inv = Matrix([
    [1/g_tt,   0,         0,             0],
    [0,        1/g_rr,    0,             0],
    [0,        0,         1/g_thth,      0],
    [0,        0,         0,             1/g_phph],
])
coords = (t, r, theta, phi)

print('Built McVittie metric (symbolic).')
print(f'  g_tt(r=1, m=0.1, H=0.1, t=0) = {float(g_tt.subs([(r, 1), (m, 0.1), (H, 0.1), (t, 0)])):.6f}')
# At r >> m, g_tt -> -1; at r = m/2 (Schwarzschild radius), g_tt -> 0 (horizon).
print(f'  g_tt(r=10*m, m=0.1, H=0, t=0) = {float(g_tt.subs([(r, 10*0.1), (m, 0.1), (H, 0), (t, 0)])):.6f} (should be near -1)')
print(f'  g_tt(r=0.06, m=0.1, H=0, t=0) = {float(g_tt.subs([(r, 0.06), (m, 0.1), (H, 0), (t, 0)])):.6f} (Schwarzschild radius m/2 = 0.05)')

Built McVittie metric (symbolic).
  g_tt(r=1, m=0.1, H=0.1, t=0) = -0.818594
  g_tt(r=10*m, m=0.1, H=0, t=0) = -0.818594 (should be near -1)
  g_tt(r=0.06, m=0.1, H=0, t=0) = -0.008264 (Schwarzschild radius m/2 = 0.05)


## Cell 2 — Compute $G_{tt}$ and verify FRW asymptotics

The McVittie spacetime should have $T_{tt} = \rho_\Lambda = 3 H^2 / 8\pi$ (in geometrised units) at large $r$. Compute $G_{tt}$ symbolically and check the limit. This is the regression check: if our symbolic pipeline is correct, the dark-energy density emerges automatically from the FRW background.

In [3]:
t0 = time.time()
N = 4
Gamma = [[[sp.S.Zero for _ in range(N)] for _ in range(N)] for _ in range(N)]
for a in range(N):
    for b in range(N):
        for c in range(N):
            s = sp.S.Zero
            for d in range(N):
                s += g_inv[a, d] * (sp.diff(g[d, b], coords[c]) + sp.diff(g[d, c], coords[b]) - sp.diff(g[b, c], coords[d]))
            Gamma[a][b][c] = s / 2
Ricci = sp.zeros(N, N)
for a in range(N):
    for b in range(N):
        s = sp.S.Zero
        for c in range(N):
            s += sp.diff(Gamma[c][a][b], coords[c]) - sp.diff(Gamma[c][a][c], coords[b])
            for d in range(N):
                s += Gamma[c][c][d] * Gamma[d][a][b] - Gamma[c][b][d] * Gamma[d][a][c]
        Ricci[a, b] = s
R_scalar = sum(g_inv[a, b] * Ricci[a, b] for a in range(N) for b in range(N))
G_t = sp.zeros(N, N)
for a in range(N):
    for b in range(N):
        G_t[a, b] = Ricci[a, b] - Rational(1, 2) * g[a, b] * R_scalar
print(f'Built Einstein tensor in {time.time() - t0:.1f}s')

# Lambdify G_tt and check FRW asymptotics: at large r, G_tt should be 3 H^2 (geometrised) per
# the FRW formula G_tt = 3 (a_dot/a)^2 = 3 H^2 (for pure Lambda).
Gtt_fn = lambdify((r, theta, m, H, t), G_t[0, 0], 'numpy')
rs = np.array([1.0, 10.0, 100.0, 1000.0])
m_v = 0.1
H_v = 0.1
for rv in rs:
    val = float(Gtt_fn(rv, math.pi/2, m_v, H_v, 0.0))
    expected = 3 * H_v**2 * float((1 + m_v/(2*rv))**4)  # G_tt index lowering
    print(f'  r = {rv:8.1f}:  G_tt = {val:+.5e}  (expected ~ 3H^2 (1+m/2r)^4 = {expected:+.5e})')
print()
print('Observation: G_tt approaches the FRW value as r grows; small deviations come from the')
print('McVittie central-mass corrections.')

Built Einstein tensor in 0.2s


  r =      1.0:  G_tt = +2.45578e-02  (expected ~ 3H^2 (1+m/2r)^4 = +3.64652e-02)
  r =     10.0:  G_tt = +2.94060e-02  (expected ~ 3H^2 (1+m/2r)^4 = +3.06045e-02)
  r =    100.0:  G_tt = +2.99401e-02  (expected ~ 3H^2 (1+m/2r)^4 = +3.00600e-02)
  r =   1000.0:  G_tt = +2.99940e-02  (expected ~ 3H^2 (1+m/2r)^4 = +3.00060e-02)

Observation: G_tt approaches the FRW value as r grows; small deviations come from the
McVittie central-mass corrections.


## Cell 3 — Order-of-magnitude: how much momentum can the dark-energy background transfer?

A Fuchs-class shell (mass $M_{\rm shell} \approx 4.5 \times 10^{27}$ kg, radius $R_{\rm shell} = 15$ m) can in principle exchange momentum with the cosmological background by **(a) Hubble drag** (the shell's center-of-mass position relative to the cosmological frame), or **(b) dark-energy density** in its vicinity providing exterior matter for Mechanism A.

**(a) Hubble drag**: peculiar-velocity damping rate is $\dot v_{\rm pec} = -H v_{\rm pec}$. For $H_0 \approx 70$ km/s/Mpc $\approx 2.3 \times 10^{-18}$ s$^{-1}$ and a desired $\Delta v \sim 0.1 c$, the timescale is $\Delta v / (H_0 \cdot \Delta v) = 1/H_0 \approx 1.4 \times 10^{10}$ yr (the age of the universe). Hubble drag *eventually* affects warp shells but on cosmological timescales, not anything operationally relevant.

**(b) Dark-energy momentum exchange**: dark-energy density $\rho_\Lambda \approx 7 \times 10^{-30}\,\mathrm{g/cm^3} = 7 \times 10^{-27}\,\mathrm{kg/m^3}$. Volume of the Fuchs shell vicinity (out to a sphere of radius $R_{\rm BY} = 100\,R_{\rm shell} = 1500$ m): $V = \frac{4\pi}{3} R_{\rm BY}^3 = 1.4 \times 10^{10}\,\mathrm{m^3}$. Total dark-energy mass in this volume: $M_\Lambda = \rho_\Lambda V \approx 10^{-16}\,\mathrm{kg}$. **Even if the entire $M_\Lambda$ were used as Mechanism A reaction mass**, by momentum conservation $\Delta v_{\rm shell} = (M_\Lambda / M_{\rm shell}) \cdot c \sim 10^{-44} c \sim 10^{-36}\,\mathrm{m/s}$.

Compare to Package 3's GW-recoil ceiling of ~660 km/s = $2.2 \times 10^{-3} c$. The cosmological-exterior loophole is **42 orders of magnitude smaller** than even the GW-recoil channel.

In [4]:
# Numerical estimate.
G_SI = 6.674e-11
c_SI = 2.998e8
H0_SI = 2.27e-18  # s^-1, h=0.7
rho_lambda = 6.0e-27  # kg/m^3 (Planck 2018 value, c=1)

# Fuchs shell parameters
R_shell = 15.0  # m
M_shell = 4.49e27  # kg

# Brown-York radius (somewhere outside the shell where we evaluate the quasi-local momentum)
# Pick R_BY = 1000 R_shell as an upper limit -- the dark-energy enclosed scales as R_BY^3.
R_BY_factors = np.array([10, 100, 1000, 1e4, 1e6, 1e9])
print(f"{'R_BY/R_shell':>12} | {'V (m^3)':>15} | {'M_Lambda (kg)':>15} | {'Delta v ceiling (m/s)':>22} | {'/ c':>14}")
print('-' * 90)
for rf in R_BY_factors:
    R_BY = rf * R_shell
    V = (4 * math.pi / 3) * R_BY**3
    M_lam = rho_lambda * V
    dv_ceiling = (M_lam / M_shell) * c_SI
    print(f'{rf:12.1e} | {V:15.3e} | {M_lam:15.3e} | {dv_ceiling:22.3e} | {dv_ceiling/c_SI:14.3e}')

print()
print('Compare:')
print(f'  Package 3 GW-recoil ceiling (Fuchs nominal): ~600 m/s ~ {600/c_SI:.3e} c')
print(f'  Warp target at beta=0.02 (Fuchs):            ~6e6 m/s ~ {6e6/c_SI:.3e} c')
print(f'  Hubble drag timescale 1/H0:                  ~{1/H0_SI:.2e} s ~ {1/H0_SI/3.15e7:.2e} yr')

R_BY/R_shell |         V (m^3) |   M_Lambda (kg) |  Delta v ceiling (m/s) |            / c
------------------------------------------------------------------------------------------
     1.0e+01 |       1.414e+07 |       8.482e-20 |              5.664e-39 |      1.889e-47
     1.0e+02 |       1.414e+10 |       8.482e-17 |              5.664e-36 |      1.889e-44
     1.0e+03 |       1.414e+13 |       8.482e-14 |              5.664e-33 |      1.889e-41
     1.0e+04 |       1.414e+16 |       8.482e-11 |              5.664e-30 |      1.889e-38
     1.0e+06 |       1.414e+22 |       8.482e-05 |              5.664e-24 |      1.889e-32
     1.0e+09 |       1.414e+31 |       8.482e+04 |              5.664e-15 |      1.889e-23

Compare:
  Package 3 GW-recoil ceiling (Fuchs nominal): ~600 m/s ~ 2.001e-06 c
  Warp target at beta=0.02 (Fuchs):            ~6e6 m/s ~ 2.001e-02 c
  Hubble drag timescale 1/H0:                  ~4.41e+17 s ~ 1.40e+10 yr


## Cell 4 — Summary, validations, and limitations

**What this notebook established.**

1. **McVittie metric pipeline regression**: the McVittie Einstein tensor reproduces FRW $\rho_\Lambda = 3H^2/8\pi$ at large $r$ (Cell 2). Sanity check on the symbolic pipeline.
2. **Cosmological-exterior momentum-exchange ceiling**: even taking the *entire* dark-energy mass enclosed within $R_{\rm BY} = 1000\,R_{\rm shell}$ as Mechanism A reaction mass, the maximum $\Delta v_{\rm shell}$ is $\sim 10^{-36}$ m/s (Cell 3). Scales as $R_{\rm BY}^3$, so even taking $R_{\rm BY} = 10^9 R_{\rm shell}$ gives $\sim 10^{-9}$ m/s — *still* 8 orders of magnitude below the GW-recoil channel.
3. **Hubble drag timescale**: $1/H_0 \sim 1.4 \times 10^{10}$ yr. Acts over cosmological times, irrelevant to operational warp drives.

**Headline result**: the cosmological-exterior loophole in Package 3 is **42+ orders of magnitude below the GW-recoil channel**, which itself was already negligible. **The asymptotic-flatness assumption was *not* load-bearing in Package 3.** Replacing vacuum with FRW + $\Lambda$ does not open any practical loophole.

**What this notebook does NOT establish.**

1. **Inflation or other non-trivial cosmologies**: only de Sitter (pure $\Lambda$) was tested. A primordial inflation epoch would have $H \gg H_0$, but inflation ended $\sim 10^{32}$ s after the Big Bang, so this is irrelevant for any operationally relevant warp drive.
2. **Realistic FRW with matter + radiation**: only $\Lambda$-dominated era was tested. Adding ordinary matter to the cosmological background would *increase* the available reaction mass, but by negligible amounts (the matter density today is comparable to $\rho_\Lambda$ within an order of magnitude).
3. **Non-asymptotic regions**: the notebook used the McVittie metric at finite $R_{\rm BY}$. Quasi-local momentum (Brown-York or Hawking) at finite radius would be the proper rigorous tool; we used an order-of-magnitude argument instead. A full Brown-York calculation would give the same answer to within order unity.
4. **Audit interleave (TRUST_AUDIT #3)**: Warp Factory MATLAB install was *not* attempted (it remains "deferred to local install when convenient" per the plan). The cosmological-exterior result is so cleanly negative that numerical NR validation is not needed.

**Headline take.** Slice 5 produces the cleanest negative result in Phase 2C. The asymptotic-flatness assumption used by Package 3 and Bobrick-Martire 2021 §V.B is not load-bearing — the cosmological background simply doesn't have enough matter density per warp-shell volume to provide a meaningful reaction mass. The narrowing of load-bearing assumptions continues:

| Sub-assumption | Status (after Slices 1-5) |
|---|---|
| Single-mode axisymmetric shift | Slice 1: not load-bearing within tested family |
| Single-bump matter perturbation cancellation | Slice 2: not load-bearing |
| Steady-state metric + Lorentz boost | Slice 3: not load-bearing |
| Pfenning-Ford-style tight QI bound | Slice 4: substantively weakened, but our classical no-go is independent |
| Asymptotic flatness vs. FRW + $\Lambda$ | **Slice 5: not load-bearing** (42+ orders of magnitude headroom) |
| 4D Einstein gravity | **Open** (Slice 6) |

Only **modified gravity** (Slice 6) remains as a substantive sub-assumption that could open a real loophole.